# 02 — Demand Forecasting

## Goals
- Load daily sales from Parquet for FOODS category only
- Engineer time series features — lags, rolling averages, calendar features
- Train LightGBM demand forecasting model
- Evaluate forecast accuracy — RMSE and MAPE per product
- Save model and predictions to disk for use in simulation

## Why Demand Forecasting First
The inventory simulation in notebook 03 needs a realistic demand signal.
Using raw historical averages ignores trends, seasonality, price effects,
and holiday patterns. A trained forecasting model produces more realistic
synthetic demand — making the simulation and causal analysis more credible.

## Scope
FOODS category only — highest velocity, most stockout sensitivity,
most price variation. 3,049 × 4 CA stores = ~12,000 product-store combinations.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Paths
PARQUET_PATH = Path('../data/parquet')
PROCESSED_PATH = Path('../data/processed')
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

# Load FOODS category, California stores only
print("Loading FOODS/CA data from Parquet...")
df = pd.read_parquet(
    PARQUET_PATH / 'daily_sales.parquet',
    filters=[
        ('cat_id', '=', 'FOODS'),
        ('state_id', '=', 'CA')
    ]
)

print(f"Shape: {df.shape}")
print(f"Unique items: {df['item_id'].nunique()}")
print(f"Unique stores: {df['store_id'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

Loading FOODS/CA data from Parquet...
Shape: (10995924, 19)
Unique items: 1437
Unique stores: 4
Date range: 2011-01-29 00:00:00 to 2016-04-24 00:00:00
Memory usage: 7482.2 MB


In [2]:
# Reduce memory usage drastically
print("Reducing memory usage...")
print(f"Before: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")

# Downcast numeric columns
df['units_sold'] = df['units_sold'].astype('int16')
df['wday'] = df['wday'].astype('int8')
df['month'] = df['month'].astype('int8')
df['year'] = df['year'].astype('int16')
df['snap_CA'] = df['snap_CA'].astype('int8')
df['snap_TX'] = df['snap_TX'].astype('int8')
df['snap_WI'] = df['snap_WI'].astype('int8')

# Convert categoricals
for col in ['item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 
            'weekday', 'event_name_1', 'event_type_1']:
    df[col] = df[col].astype('category')

# Drop columns we don't need for forecasting
df = df.drop(columns=['id', 'cat_id', 'state_id', 'd', 'wm_yr_wk', 
                       'snap_TX', 'snap_WI', 'event_type_1'])

print(f"After:  {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
print(f"Shape:  {df.shape}")
print(f"Columns: {df.columns.tolist()}")

Reducing memory usage...
Before: 7482.2 MB
After:  220.3 MB
Shape:  (10995924, 11)
Columns: ['item_id', 'dept_id', 'store_id', 'units_sold', 'date', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'snap_CA']


In [3]:
# Feature engineering — lag and rolling features
print("Engineering features...")

# Sort by item, store, date — critical for lag features
df = df.sort_values(['item_id', 'store_id', 'date']).reset_index(drop=True)

# Group key for lag calculations
group = df.groupby(['item_id', 'store_id'])['units_sold']

# Lag features — what sold N days ago
print("  Computing lag features...")
df['lag_7']  = group.shift(7)   # same day last week
df['lag_14'] = group.shift(14)  # two weeks ago
df['lag_28'] = group.shift(28)  # four weeks ago

# Rolling mean features — smooth demand signal
print("  Computing rolling features...")
df['roll_mean_7']  = group.shift(1).transform(lambda x: x.rolling(7,  min_periods=1).mean())
df['roll_mean_28'] = group.shift(1).transform(lambda x: x.rolling(28, min_periods=1).mean())
df['roll_std_7']   = group.shift(1).transform(lambda x: x.rolling(7,  min_periods=1).std())

# Drop rows with NaN lags — first 28 days per product
df = df.dropna(subset=['lag_7', 'lag_14', 'lag_28'])

# Encode categoricals as integer codes
df['item_id']      = df['item_id'].cat.codes
df['dept_id']      = df['dept_id'].cat.codes
df['store_id']     = df['store_id'].cat.codes
df['event_name_1'] = df['event_name_1'].cat.codes

print(f"Shape after feature engineering: {df.shape}")
print(f"Memory: {df.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
print(f"\nFeature columns:")
print(df.columns.tolist())

Engineering features...
  Computing lag features...
  Computing rolling features...
Shape after feature engineering: (10834980, 17)
Memory: 795.6 MB

Feature columns:
['item_id', 'dept_id', 'store_id', 'units_sold', 'date', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'snap_CA', 'lag_7', 'lag_14', 'lag_28', 'roll_mean_7', 'roll_mean_28', 'roll_std_7']


In [4]:
# Train/validation split — last 28 days as validation
# This is a time-based split — never use future data to predict the past
CUTOFF_DATE = pd.Timestamp('2016-03-27')  # 28 days before end of data

feature_cols = ['item_id', 'dept_id', 'store_id', 'wday', 'month', 'year',
                'event_name_1', 'snap_CA', 'lag_7', 'lag_14', 'lag_28',
                'roll_mean_7', 'roll_mean_28', 'roll_std_7']

target_col = 'units_sold'

train = df[df['date'] <= CUTOFF_DATE]
valid = df[df['date'] > CUTOFF_DATE]

print(f"Train: {len(train):,} rows ({train['date'].min().date()} to {train['date'].max().date()})")
print(f"Valid: {len(valid):,} rows ({valid['date'].min().date()} to {valid['date'].max().date()})")

X_train = train[feature_cols]
y_train = train[target_col]
X_valid = valid[feature_cols]
y_valid = valid[target_col]

# Train LightGBM
print("\nTraining LightGBM...")
dtrain = lgb.Dataset(X_train, label=y_train)
dvalid = lgb.Dataset(X_valid, label=y_valid, reference=dtrain)

params = {
    'objective':        'tweedie',   # good for count data with zeros
    'tweedie_variance_power': 1.1,
    'metric':           'rmse',
    'learning_rate':    0.05,
    'num_leaves':       63,
    'min_child_samples': 20,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq':     1,
    'verbose':          -1
}

callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=50)
]

model = lgb.train(
    params,
    dtrain,
    num_boost_round=500,
    valid_sets=[dvalid],
    callbacks=callbacks
)

print(f"\nBest iteration: {model.best_iteration}")

Train: 10,674,036 rows (2011-02-26 to 2016-03-27)
Valid: 160,944 rows (2016-03-28 to 2016-04-24)

Training LightGBM...
Training until validation scores don't improve for 50 rounds
[50]	valid_0's rmse: 2.33696
[100]	valid_0's rmse: 2.29982
[150]	valid_0's rmse: 2.29272
[200]	valid_0's rmse: 2.28928
[250]	valid_0's rmse: 2.28704
[300]	valid_0's rmse: 2.28538
[350]	valid_0's rmse: 2.28476
[400]	valid_0's rmse: 2.28363
[450]	valid_0's rmse: 2.28294
[500]	valid_0's rmse: 2.28195
Did not meet early stopping. Best iteration is:
[500]	valid_0's rmse: 2.28195

Best iteration: 500


In [5]:
# Evaluate model
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Predictions on validation set
valid_preds = model.predict(X_valid, num_iteration=model.best_iteration)
valid_preds = np.clip(valid_preds, 0, None)  # no negative sales

# Metrics
rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
mae  = mean_absolute_error(y_valid, valid_preds)
# MAPE only on non-zero actuals
mask = y_valid > 0
mape = np.mean(np.abs((y_valid[mask] - valid_preds[mask]) / y_valid[mask])) * 100

print("=== VALIDATION METRICS ===")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"MAPE: {mape:.2f}% (on non-zero sales days)")
print(f"\nMean actual sales:    {y_valid.mean():.4f}")
print(f"Mean predicted sales: {valid_preds.mean():.4f}")
print(f"Zero sales days:      {(y_valid == 0).mean()*100:.1f}%")

# Feature importance
print("\n=== FEATURE IMPORTANCE (top 10) ===")
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)
print(importance.head(10).to_string(index=False))

=== VALIDATION METRICS ===
RMSE: 2.2819
MAE:  1.2693
MAPE: 53.02% (on non-zero sales days)

Mean actual sales:    2.0243
Mean predicted sales: 2.0369
Zero sales days:      44.5%

=== FEATURE IMPORTANCE (top 10) ===
     feature   importance
 roll_mean_7 1.871794e+08
roll_mean_28 1.226525e+08
  roll_std_7 1.281987e+07
        wday 2.591732e+06
      lag_14 1.151406e+06
event_name_1 8.515606e+05
       lag_7 8.514984e+05
      lag_28 8.099703e+05
     item_id 7.321293e+05
        year 4.998304e+05


In [6]:
# Continue training for more rounds
print("Extending training to 1000 rounds...")

model = lgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    valid_sets=[dvalid],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=True),
        lgb.log_evaluation(period=100)
    ]
)

# Re-evaluate
valid_preds = np.clip(model.predict(X_valid), 0, None)
rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
mae  = mean_absolute_error(y_valid, valid_preds)
mape = np.mean(np.abs((y_valid[mask] - valid_preds[mask]) / y_valid[mask])) * 100

print(f"\nUpdated metrics:")
print(f"RMSE: {rmse:.4f}")
print(f"MAE:  {mae:.4f}")
print(f"MAPE: {mape:.2f}%")
print(f"Best iteration: {model.best_iteration}")

Extending training to 1000 rounds...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 2.29982
[200]	valid_0's rmse: 2.28928
[300]	valid_0's rmse: 2.28538
[400]	valid_0's rmse: 2.28363
[500]	valid_0's rmse: 2.28195
Early stopping, best iteration is:
[519]	valid_0's rmse: 2.28147

Updated metrics:
RMSE: 2.2815
MAE:  1.2693
MAPE: 53.02%
Best iteration: 519


In [7]:
import joblib

# Save model
MODEL_PATH = Path('../outputs')
MODEL_PATH.mkdir(parents=True, exist_ok=True)

model.save_model(str(MODEL_PATH / 'lgb_demand_model.txt'))
print(f"Model saved to outputs/lgb_demand_model.txt")

# Save validation predictions
valid_results = valid[['item_id', 'store_id', 'date', 'units_sold']].copy()
valid_results['predicted'] = valid_preds
valid_results.to_parquet(PARQUET_PATH / 'validation_predictions.parquet', index=False)
print(f"Validation predictions saved to parquet/validation_predictions.parquet")

# Save feature list for downstream notebooks
import json
with open(MODEL_PATH / 'feature_cols.json', 'w') as f:
    json.dump(feature_cols, f)
print(f"Feature columns saved to outputs/feature_cols.json")

# Save full predictions for simulation
print("\nGenerating predictions for full dataset...")
full_preds = np.clip(model.predict(df[feature_cols]), 0, None)
df['demand_forecast'] = full_preds

forecast_df = df[['item_id', 'store_id', 'date', 'units_sold', 'demand_forecast']].copy()
forecast_df.to_parquet(PARQUET_PATH / 'demand_forecasts.parquet', index=False)
print(f"Full forecasts saved — shape: {forecast_df.shape}")
print(f"Size on disk: {(PARQUET_PATH / 'demand_forecasts.parquet').stat().st_size / 1024 / 1024:.1f} MB")

Model saved to outputs/lgb_demand_model.txt
Validation predictions saved to parquet/validation_predictions.parquet
Feature columns saved to outputs/feature_cols.json

Generating predictions for full dataset...
Full forecasts saved — shape: (10834980, 5)
Size on disk: 100.7 MB


## Summary

### Model
- **Algorithm:** LightGBM with Tweedie objective (suited for count data with zeros)
- **Best iteration:** 519 rounds
- **Training data:** 10.67M rows (2011-02-26 to 2016-03-27)
- **Validation data:** 160K rows (2016-03-28 to 2016-04-24)

### Performance
| Metric | Value |
|---|---|
| RMSE | 2.2815 |
| MAE | 1.2693 |
| MAPE | 53.02% (non-zero days only) |
| Mean actual sales | 2.02 units/day |
| Zero sales days | 44.5% |

### Key findings
- Rolling mean features dominate — recent demand history is the strongest predictor
- Day of week is the 4th most important feature — weekend patterns are significant
- 44.5% zero sales days confirms intermittent demand — expected for retail at daily granularity
- MAPE is high but misleading — unreliable when actuals are small integers
- Model converged at 519 rounds — additional training yields diminishing returns

### Outputs saved
- `outputs/lgb_demand_model.txt` — trained LightGBM model
- `outputs/feature_cols.json` — feature column list for downstream notebooks
- `data/parquet/demand_forecasts.parquet` — 10.8M rows of forecasts + actuals
- `data/parquet/validation_predictions.parquet` — held-out validation predictions

### Next
`03_inventory_simulation.ipynb` — use demand forecasts to simulate warehouse
inventory dynamics and generate stockout and holding cost outcomes